In [1]:
import pandas as pd
import re
import ipywidgets as W
from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional, Set, Dict
import re, shutil
from pathlib import Path


In [2]:
# -------------------- Column pickers (robust) --------------------

def pick_exact(df: pd.DataFrame, names: list[str]) -> str:
    """Return the first column whose name equals (case-insensitively) one of `names`."""
    lower_map = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in lower_map:
            return lower_map[n.lower()]
    raise KeyError(f"None of the exact names {names} found. Columns present: {list(df.columns)}")

def pick_contains(df: pd.DataFrame, include_terms: list[str], exclude_terms: list[str] = None) -> str:
    """Return the first column that contains any include term (ci) and none of the exclude terms."""
    exclude_terms = exclude_terms or []
    for c in df.columns:
        cl = c.lower()
        if any(t.lower() in cl for t in include_terms) and not any(x.lower() in cl for x in exclude_terms):
            return c
    raise KeyError(f"No column matched include={include_terms} exclude={exclude_terms}. Columns: {list(df.columns)}")

# -------------------- Parsers --------------------

def load_book_base(book_csv: str) -> tuple[pd.DataFrame, str, str]:
    """
    Expect columns including EXACT 'Sequence' (NOT 'Sequence Name') and a 'Well' column.
    Returns tidy df with columns: Well, Sequence; also returns the chosen column names.
    """
    df = pd.read_csv(book_csv)

    # Well column: prefer 'Well Position', else 'Well'
    try:
        well_col = pick_exact(df, ["Well Position"])
    except KeyError:
        well_col = pick_exact(df, ["Well"])

    # Sequence column in book_base MUST be EXACT 'Sequence'
    seq_col = pick_exact(df, ["Sequence"])   # <-- never matches 'Sequence Name'

    tidy = pd.DataFrame({
        "Well": df[well_col].astype(str).str.upper().str.strip(),
        "Sequence": df[seq_col].astype(str).str.replace(" ", "", regex=False).str.upper().str.strip(),
    })
    tidy = tidy[(tidy["Well"]!="") & (tidy["Sequence"]!="")].reset_index(drop=True)
    return tidy, well_col, seq_col

def load_replace(replace_csv: str) -> tuple[pd.DataFrame, str, str]:
    """
    Expect 'Well' and a sequence column (e.g., 'Sequence' or "Sequence (5' > 3')").
    Returns tidy df with Well, Sequence; also returns chosen column names.
    """
    df = pd.read_csv(replace_csv)
    well_col = pick_exact(df, ["Well"])  # replace sheet uses 'Well'
    # Accept either exact 'Sequence' or the 5'>3' variant; never 'Sequence Name'
    try:
        seq_col = pick_exact(df, ["Sequence"])
    except KeyError:
        seq_col = pick_contains(df, ["sequence"], exclude_terms=["name"])  # avoids 'Sequence Name'

    tidy = pd.DataFrame({
        "Well": df[well_col].astype(str).str.upper().str.strip(),
        "Sequence": df[seq_col].astype(str).str.replace(" ", "", regex=False).str.upper().str.strip(),
    })
    tidy = tidy[(tidy["Well"]!="") & (tidy["Sequence"]!="")].reset_index(drop=True)
    return tidy, well_col, seq_col

# -------------------- Sequence utilities --------------------

def revcomp(seq: str) -> str:
    comp = str.maketrans("ACGTN", "TGCAN")
    return seq.translate(comp)[::-1]

def lcs_substring_with_pos(a: str, b: str):
    """Longest common *substring* (contiguous) length and index ranges in a and b."""
    if not a or not b:
        return 0, None, None
    la, lb = len(a), len(b)
    prev = [0]*(lb+1)
    best = 0
    apos = None
    bpos = None
    for i in range(1, la+1):
        curr = [0]*(lb+1)
        ai = a[i-1]
        for j in range(1, lb+1):
            if ai == b[j-1]:
                curr[j] = prev[j-1] + 1
                if curr[j] > best:
                    best = curr[j]
                    a_end = i
                    b_end = j
                    apos = (a_end - best, a_end)
                    bpos = (b_end - best, b_end)
        prev = curr
    return best, apos, bpos

def lcs_subsequence_len(a: str, b: str) -> int:
    """Longest common *subsequence* (not necessarily contiguous)."""
    la, lb = len(a), len(b)
    dp = [0]*(lb+1)
    for i in range(1, la+1):
        prev = 0
        ai = a[i-1]
        for j in range(1, lb+1):
            tmp = dp[j]
            if ai == b[j-1]:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j-1])
            prev = tmp
    return dp[-1]

# -------------------- Matching + metrics + viz --------------------

def compute_matches_with_metrics(book_csv: str, replace_csv: str, min_overlap: int = 1) -> pd.DataFrame:
    book, book_well_col, book_seq_col = load_book_base(book_csv)
    repl, repl_well_col, repl_seq_col = load_replace(replace_csv)

    print(f"[book_base] using Well='{book_well_col}', Sequence='{book_seq_col}' (EXACT match).")
    print(f"[replace]   using Well='{repl_well_col}', Sequence='{repl_seq_col}'.")

    # Previews to double-check parsing
    print("\nBook base (Well, Sequence) preview:")
    display(book.head(10))
    print("\nReplace (Well, Sequence) preview:")
    display(repl.head(10))

    rows = []
    # cache
    b_wells = book["Well"].tolist()
    b_seqs  = book["Sequence"].tolist()

    for _, r in repl.iterrows():
        rwell = r["Well"]
        rseq  = r["Sequence"]
        rseq_rc = revcomp(rseq)

        best = {"len": -1}

        # try forward
        for bw, bs in zip(b_wells, b_seqs):
            L, apos, bpos = lcs_substring_with_pos(rseq, bs)
            if L > best["len"]:
                best = {"len": L, "bw": bw, "bs": bs, "apos": apos, "bpos": bpos, "ori": "fwd"}

        # try reverse complement
        for bw, bs in zip(b_wells, b_seqs):
            L, apos, bpos = lcs_substring_with_pos(rseq_rc, bs)
            if L > best["len"]:
                best = {"len": L, "bw": bw, "bs": bs, "apos": apos, "bpos": bpos, "ori": "revcomp"}

        if best["len"] >= min_overlap and best["bw"] is not None:
            # extract the matched substring as it appears in the replacement seq
            match_sub = ""
            if best["apos"]:
                a_start, a_end = best["apos"]
                if best["ori"] == "fwd":
                    match_sub = rseq[a_start:a_end]
                else:
                    rc_sub = rseq_rc[a_start:a_end]
                    match_sub = revcomp(rc_sub)  # show in replace orientation

            r_len = len(rseq)
            b_len = len(best["bs"])
            rows.append({
                "Replace_Well": rwell,
                "Replace_Len": r_len,
                "Best_Orientation": best["ori"],
                "Matched_Base_Well": best["bw"],
                "Base_Len": b_len,
                "Overlap_Len": best["len"],
                "Overlap_Frac_of_Replace": best["len"]/r_len if r_len else 0.0,
                "Overlap_Frac_of_Base": best["len"]/b_len if b_len else 0.0,
                "Symmetric_Overlap": (2*best["len"])/(r_len+b_len) if (r_len+b_len) else 0.0,
                "LCS_Subsequence_Len": lcs_subsequence_len(rseq if best["ori"]=="fwd" else rseq_rc, best["bs"]),
                "Match_Substring": match_sub,
            })
        else:
            rows.append({
                "Replace_Well": rwell,
                "Replace_Len": len(rseq),
                "Best_Orientation": None,
                "Matched_Base_Well": None,
                "Base_Len": None,
                "Overlap_Len": 0,
                "Overlap_Frac_of_Replace": 0.0,
                "Overlap_Frac_of_Base": 0.0,
                "Symmetric_Overlap": 0.0,
                "LCS_Subsequence_Len": 0,
                "Match_Substring": "",
            })

    df = pd.DataFrame(rows).sort_values("Overlap_Len", ascending=False).reset_index(drop=True)

    # Save + quick visuals
    df.to_csv("match_metrics.csv", index=False)
    print("\nSaved full metrics to match_metrics.csv")

    if not df.empty:
        ax = df["Overlap_Len"].plot.hist(bins=20, title="Histogram of Overlap Lengths")
        ax.set_xlabel("Overlap length (longest common substring)")
        plt.show()

        plt.figure()
        plt.scatter(df["Replace_Len"], df["Overlap_Frac_of_Replace"])
        plt.title("Overlap Fraction vs Replace Length")
        plt.xlabel("Replace length")
        plt.ylabel("Overlap fraction (replace)")
        plt.show()

    return df

# -------------------- Optional helper: write Replace Well back --------------------

def write_replace_well(replace_csv: str, matches_df: pd.DataFrame, out_csv: str | None = None):
    """
    Adds a 'Replace Well' column to the replace CSV using Matched_Base_Well.
    If out_csv is None, overwrites replace_csv; else writes to out_csv.
    """
    df = pd.read_csv(replace_csv)
    # Map by the file's actual 'Well' column name
    well_col = pick_exact(df, ["Well"])
    mapping = dict(zip(matches_df["Replace_Well"], matches_df["Matched_Base_Well"]))
    df["Replace Well"] = df[well_col].astype(str).str.upper().map(mapping)
    target = out_csv or replace_csv
    df.to_csv(target, index=False)
    print(f"Wrote 'Replace Well' to: {target}")

# -------------------- Example run --------------------
# book_path = "book_base.csv"
# repl_path = "yaritza_replace.csv"
# metrics_df = compute_matches_with_metrics(book_path, repl_path)
# write_replace_well(repl_path, metrics_df)  # optional


In [3]:
import re, shutil
from pathlib import Path
from datetime import datetime
import pandas as pd

def rename_setc_inplace(csv_path: str):
    """
    In-place renamer for Set C strands in a CSV that has a preamble row block and a
    true header ('Well', 'Name', ...) several rows down.

    Left-side mapping:
      - l[1-5]_aptamer_pdgf          -> PDGF-Apt_<Top/Mid/Bot>-<L3/L2/L1>
      - l[1-5]_aptamer_kanamycin     -> Kana-Apt_<Top/Mid/Bot>-<L3/L2/L1>

    Right-side mapping (complements):
      - l[1-5]_aptamer_pdgf_complement_14bp/18bp/22bp/26bp/30bp/34bp/38bp
           -> PDGF-<len>_<Top/Mid/Bot>-<R3/R2/R1>
      - l[1-5]_aptamer_kanamycin_complement_14bp/18bp/22bp
           -> Kana-<len>_<Top/Mid/Bot>-<R3/R2/R1>

    Returns: (header_row_index, name_col_index, considered_count, renamed_count)
    """
    p = Path(csv_path)
    if not p.exists():
        raise FileNotFoundError(f"File not found: {p}")

    # Read raw (no header), preserve everything exactly
    raw = pd.read_csv(p, header=None, dtype=object, engine="python")
    n_rows, n_cols = raw.shape

    def norm(x):
        # normalize for matching; keep original cells untouched
        return ("" if pd.isna(x) else str(x).replace("\u00A0"," ").strip().lower())

    # ---- find real header row (first row that has both 'well' and 'name') ----
    header_idx = None
    for i in range(n_rows):
        vals = [norm(v) for v in raw.iloc[i].tolist()]
        if "well" in vals and "name" in vals:
            header_idx = i
            break
    if header_idx is None:
        raise ValueError("Could not find a header row containing both 'Well' and 'Name'.")

    # ---- find Name column index within that header row ----
    name_col_idx = None
    for j in range(n_cols):
        if norm(raw.iat[header_idx, j]) == "name":
            name_col_idx = j
            break
    if name_col_idx is None:
        raise ValueError("Name column not found in the detected header row.")

    # ---- patterns observed in your file ----
    # allow separators: _, -, or space
    sep = r"[\s_\-]"
    pat_left_pdgf = re.compile(rf"^l([1-5]){sep}aptamer{sep}pdgf$", re.IGNORECASE)
    pat_left_kana = re.compile(rf"^l([1-5]){sep}aptamer{sep}kanamycin$", re.IGNORECASE)
    pat_right_pdgf = re.compile(rf"^l([1-5]){sep}aptamer{sep}pdgf{sep}complement{sep}(\d{{2}})bp$", re.IGNORECASE)
    pat_right_kana = re.compile(rf"^l([1-5]){sep}aptamer{sep}kanamycin{sep}complement{sep}(\d{{2}})bp$", re.IGNORECASE)

    # position maps
    LEFT_POS  = {"1": ("Top","L3"), "2": ("Mid","L3"), "3": ("Bot","L3"), "4": ("Mid","L2"), "5": ("Mid","L1")}
    RIGHT_POS = {"1": ("Top","R3"), "2": ("Mid","R3"), "3": ("Bot","R3"), "4": ("Mid","R2"), "5": ("Mid","R1")}

    allowed_pd = {"14","18","22","26","30","34","38"}
    allowed_ka = {"14","18","22"}

    # ---- run renaming only on rows below the header ----
    considered = 0
    renamed = 0
    for r in range(header_idx + 1, n_rows):
        old_val = raw.iat[r, name_col_idx]
        s = norm(old_val)
        if not s:
            continue

        # Left PDGF
        m = pat_left_pdgf.match(s)
        if m:
            pos = m.group(1)
            rlab, clab = LEFT_POS[pos]
            new = f"PDGF-Apt_{rlab}-{clab}"
            if new != old_val:
                raw.iat[r, name_col_idx] = new; renamed += 1
            considered += 1
            continue

        # Left Kana
        m = pat_left_kana.match(s)
        if m:
            pos = m.group(1)
            rlab, clab = LEFT_POS[pos]
            new = f"Kana-Apt_{rlab}-{clab}"
            if new != old_val:
                raw.iat[r, name_col_idx] = new; renamed += 1
            considered += 1
            continue

        # Right PDGF complements
        m = pat_right_pdgf.match(s)
        if m:
            pos, num = m.group(1), m.group(2)
            if num in allowed_pd:
                rlab, clab = RIGHT_POS[pos]
                new = f"PDGF-{num}_{rlab}-{clab}"
                if new != old_val:
                    raw.iat[r, name_col_idx] = new; renamed += 1
                considered += 1
            continue

        # Right Kana complements
        m = pat_right_kana.match(s)
        if m:
            pos, num = m.group(1), m.group(2)
            if num in allowed_ka:
                rlab, clab = RIGHT_POS[pos]
                new = f"Kana-{num}_{rlab}-{clab}"
                if new != old_val:
                    raw.iat[r, name_col_idx] = new; renamed += 1
                considered += 1
            continue

    # ---- backup + write back in place (preamble preserved) ----
    ts = datetime.now().strftime("%Y%m%d-%H%M%S")
    backup = p.with_suffix(p.suffix + f".bak-{ts}")
    shutil.copy2(p, backup)
    raw.to_csv(p, header=False, index=False)

    print(f"[OK] Considered {considered} Set C rows; renamed {renamed}.")
    print(f"[Backup] Original saved as: {backup}")
    return header_idx, name_col_idx, considered, renamed


In [4]:
import re
from pathlib import Path
import pandas as pd

def reformat_setB_names(input_csv_path: str,
                        output_csv_path: str | None = None,
                        name_col: str = "Name",
                        strict: bool = False):
    """
    Rewrite Set B sequence names into the UI format:
      "<Panel>_R<row>-<col2d>", where Panel ∈
        {"D-Orthogonal","U-Orthogonal","D-Identical","U-Identical"}.

    Input names look like "1U1_5'_Ortho", "2D4_5'_Iden", etc.
    Mapping rules (as provided):
      group=1, U or D: 1..6 -> (1,1),(1,2),(2,1),(2,2),(2,1),(2,2)
      group=2, U or D: 1..6 -> (1,3),(1,4),(2,3),(2,4),(2,3),(2,4)

    Args:
      input_csv_path: path to samuel_replace.csv
      output_csv_path: path to write; defaults to "<stem>_ui_names.csv" in same folder
      name_col: column containing the names to transform
      strict: if True, raise on any row that cannot be parsed; if False, leave such rows unchanged

    Returns:
      (output_csv_path, changed_count, total_rows)
    """
    path = Path(input_csv_path)
    if not path.exists():
        raise FileNotFoundError(f"Input CSV not found: {input_csv_path}")

    df = pd.read_csv(path)

    # Resolve name column (case-insensitive fallback)
    if name_col not in df.columns:
        lower_map = {c.lower(): c for c in df.columns}
        if "name" in lower_map:
            name_col = lower_map["name"]
        else:
            raise ValueError(f"Could not find a 'Name' column. Columns: {df.columns.tolist()}")

    # Row/col mapping tables per your rules
    base_map_1 = {1:(1,1), 2:(1,2), 3:(2,1), 4:(2,2), 5:(2,1), 6:(2,2)}
    base_map_2 = {1:(1,3), 2:(1,4), 3:(2,3), 4:(2,4), 5:(2,3), 6:(2,4)}

    pos_map = {}
    pos_map.update({f"1U{k}": v for k, v in base_map_1.items()})
    pos_map.update({f"1D{k}": v for k, v in base_map_1.items()})
    pos_map.update({f"2U{k}": v for k, v in base_map_2.items()})
    pos_map.update({f"2D{k}": v for k, v in base_map_2.items()})

    # Parse leading token like "1U1" / "2D4"
    prefix_re = re.compile(r"^\s*([12])\s*([UD])\s*([1-6])", re.IGNORECASE)

    def _panel_label(ud: str, text: str) -> str | None:
        if re.search(r"ortho", text, re.IGNORECASE):
            fam = "Orthogonal"
        elif re.search(r"iden", text, re.IGNORECASE):
            fam = "Identical"
        else:
            return None
        return f"{ud.upper()}-{fam}"

    def _compute_new_name(old_name: str) -> str | None:
        if not isinstance(old_name, str):
            return None
        m = prefix_re.search(old_name)
        if not m:
            return None
        grp, ud, idx = m.group(1), m.group(2).upper(), int(m.group(3))
        key = f"{grp}{ud}{idx}"
        if key not in pos_map:
            return None
        row, col = pos_map[key]
        panel = _panel_label(ud, old_name)
        if panel is None:
            return None
        return f"{panel}_R{row}-{col:02d}"

    # Preserve original names
    backup_col = "Old Name"
    if backup_col in df.columns:
        i = 2
        while f"{backup_col} ({i})" in df.columns:
            i += 1
        backup_col = f"{backup_col} ({i})"
    df[backup_col] = df[name_col].astype(str)

    # Transform
    changed = 0
    new_names = []
    errors = []
    for i, val in enumerate(df[name_col].astype(str)):
        nn = _compute_new_name(val)
        if nn is None:
            if strict:
                errors.append((i, val))
                new_names.append(val)
            else:
                new_names.append(val)  # leave unchanged
        else:
            new_names.append(nn)
            changed += 1
    if strict and errors:
        # Show a few offending examples for quick debugging
        examples = ", ".join(repr(v) for _, v in errors[:5])
        raise ValueError(f"{len(errors)} row(s) could not be renamed. First few: {examples}")

    df[name_col] = new_names

    # Write output
    if output_csv_path is None:
        output_csv_path = str(path.with_name(path.stem + "_ui_names.csv"))
    df.to_csv(output_csv_path, index=False)

    return output_csv_path, changed, len(df)


In [5]:
# Labcyte Echo picklist concatenation utility
# - Accepts N input CSVs and one output path
# - Unifies headers (adds missing columns as empty), keeps file provenance
# - Optional de-duplication and sorting
# - Prints a short summary and saves the merged picklist

from pathlib import Path
from typing import List, Optional
import pandas as pd

def concat_picklists(
    input_paths: List[str],
    save_path: str,
    dedupe: bool = True,
    sort: bool = True,
    keep_provenance: bool = True,
    preferred_order: Optional[List[str]] = None,
) -> pd.DataFrame:
    """
    Concatenate multiple Labcyte Echo picklist CSVs into one.
    
    Parameters
    ----------
    input_paths : list of str
        Paths to input picklist CSV files.
    save_path : str
        Output CSV path for the concatenated picklist.
    dedupe : bool, default True
        If True, drop exact duplicate transfers across files.
    sort : bool, default True
        If True, sort by common Echo columns (only those present).
    keep_provenance : bool, default True
        If True, adds an 'InputFile' column to indicate the source CSV.
    preferred_order : list of str or None
        If provided, attempt to order columns by this list first, then append any remaining.
        If None, a sensible default Echo order is used when possible.

    Returns
    -------
    df_out : pandas.DataFrame
        The merged DataFrame that was written to save_path.
    """
    # A common Echo picklist header (superset). Your files may use a subset.
    default_preferred_order = [
        # Source info
        "Source Plate Name", "Source Plate Barcode", "Source Plate Type",
        "Source Well", "Source Labware Type",
        # Destination info
        "Destination Plate Name", "Destination Plate Barcode", "Destination Plate Type",
        "Destination Well", "Destination Labware Type",
        # Transfer info
        "Transfer Volume", "Volume", "Volume (nL)",
        "Fluid Class", "Sample ID", "Description", "Group", "Comment"
    ]
    if preferred_order is None:
        preferred_order = default_preferred_order

    frames = []
    per_file_counts = []

    for p in input_paths:
        pth = Path(p)
        if not pth.exists():
            raise FileNotFoundError(f"Input file not found: {pth}")

        # Read CSV (robust to blank lines); don't auto-convert dtypes aggressively
        df = pd.read_csv(pth, dtype=str, keep_default_na=False)
        # Normalize column names: strip whitespace and collapse internal spaces
        df.columns = [c.strip().replace("\u00A0", " ") for c in df.columns]  # remove non-breaking spaces
        df.columns = [" ".join(c.split()) for c in df.columns]               # collapse repeated spaces

        # Trim string cells, but keep empties as ""
        df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

        if keep_provenance:
            df["InputFile"] = pth.name

        per_file_counts.append((pth.name, len(df)))
        frames.append(df)

    if not frames:
        raise ValueError("No input frames were loaded. Check your input_paths.")

    # Union of all columns across files
    all_cols = []
    seen = set()
    for df in frames:
        for c in df.columns:
            if c not in seen:
                seen.add(c)
                all_cols.append(c)

    # Ensure each frame has all columns (fill missing with empty string)
    unified = []
    for df in frames:
        missing = [c for c in all_cols if c not in df.columns]
        if missing:
            for c in missing:
                df[c] = ""
        # Reindex to consistent order (initially by first-seen)
        df = df[all_cols]
        unified.append(df)

    # Concatenate
    df_out = pd.concat(unified, ignore_index=True)

    # Optional de-duplication (ignore provenance when checking duplicates so identical rows merge)
    if dedupe:
        if "InputFile" in df_out.columns:
            dedupe_cols = [c for c in df_out.columns if c != "InputFile"]
        else:
            dedupe_cols = list(df_out.columns)
        before = len(df_out)
        df_out = df_out.drop_duplicates(subset=dedupe_cols, keep="first").reset_index(drop=True)
        removed = before - len(df_out)
    else:
        removed = 0

    # Column ordering: put preferred headers first (in that order) if present; then the rest
    present_preferred = [c for c in preferred_order if c in df_out.columns]
    remaining = [c for c in df_out.columns if c not in present_preferred]
    # Keep InputFile at the end (if present), for readability
    if "InputFile" in remaining:
        remaining = [c for c in remaining if c != "InputFile"] + ["InputFile"]
    df_out = df_out[present_preferred + remaining]

    # Optional sorting by common Echo keys (only apply keys that exist)
    if sort:
        sort_keys = [
            "Destination Plate Name", "Destination Plate Barcode", "Destination Well",
            "Source Plate Name", "Source Plate Barcode", "Source Well"
        ]
        sort_keys = [k for k in sort_keys if k in df_out.columns]
        if sort_keys:
            df_out = df_out.sort_values(by=sort_keys, kind="stable").reset_index(drop=True)

    # Save
    out_path = Path(save_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df_out.to_csv(out_path, index=False)

    # Summary
    total_in = sum(n for _, n in per_file_counts)
    print("=== Picklist Concatenation Summary ===")
    for name, n in per_file_counts:
        print(f"  {name:>30} : {n:,} rows")
    print("--------------------------------------")
    print(f"   Total input rows          : {total_in:,}")
    print(f"   Removed duplicate rows    : {removed:,}")
    print(f"   Final concatenated rows   : {len(df_out):,}")
    print(f"   Saved to                  : {out_path.resolve()}")

    return df_out

In [13]:
# ================= Echo Picklist GUI: A(2) / B(4) / C(12) panels + MIXING =================
# One combined picklist across all panels with duplicate Replace-Well checks
# Adds reagent-mixing UI + CSV export driven by the generated picklist
# Adds independent spacing controls for dense grids (row / col / divider)
# ===========================================================================================

from typing import Tuple, List, Dict, Set, Optional
import re
import pandas as pd
import ipywidgets as W
from IPython.display import display, Markdown

# -------- Global Base (Source1) --------
BASE_SOURCE1_DEFAULT = r"/Users/maxladabaum/Desktop/origami_book/picklists/replacement_sheets/book_base.csv"

# -------- Replacement CSVs per set --------
SOURCE2_A_DEFAULT = r"/Users/maxladabaum/Desktop/origami_book/picklists/replacement_sheets/yaritza_replace.csv"
SOURCE2_B_DEFAULT = r"/Users/maxladabaum/Desktop/origami_book/picklists/replacement_sheets/samuel_replace.csv"
SOURCE2_C_DEFAULT = r"/Users/maxladabaum/Desktop/origami_book/picklists/replacement_sheets/max_replace.csv"
SOURCE2_D_DEFAULT = r"/Users/maxladabaum/Desktop/origami_book/picklists/replacement_sheets/max_replace_MB.csv"
SOURCE2_E_DEFAULT = r"/Users/maxladabaum/Desktop/origami_book/picklists/replacement_sheets/PAINT_replace.csv"

# -------- Outputs --------
OUT_COMBINED_DEFAULT = r"/Users/maxladabaum/Desktop/origami_book/picklists/picklist_combined.csv"
OUT_MIXING_DEFAULT   = r"/Users/maxladabaum/Desktop/origami_book/picklists/mixing_recipe.csv"

# ======================== Small helpers =====================================

def _parse_vec(text: str, n: int, name: str) -> List[float]:
    try:
        vals = [float(x.strip()) for x in text.split(",")]
    except Exception:
        raise ValueError(f"Could not parse {name}. Provide {n} comma-separated numbers.")
    if len(vals) != n:
        raise ValueError(f"{name} must have exactly {n} numbers (got {len(vals)}).")
    return vals

def _link(path: str) -> str:
    return f"<a href='sandbox:{path}' target='_blank'>{path}</a>"

# =================== Robust parsers (handle preambles) =======================

def parse_source1(path: str) -> Tuple[pd.DataFrame, str]:
    raw = pd.read_csv(path, header=None)
    plate_name = None
    for i in range(5):
        if i < len(raw):
            joined = " ".join(raw.iloc[i].dropna().astype(str))
            if joined.strip().startswith("Plate "):
                plate_name = joined.strip().replace("Plate ", "")
                break
    if not plate_name:
        plate_name = "SourcePlate1[1]"
    hdr_idx = None
    for i in range(len(raw)):
        row = raw.iloc[i].astype(str).str.strip().str.upper()
        if ("WELL POSITION" in row.values) and ("SEQUENCE NAME" in row.values):
            hdr_idx = i
            break
    if hdr_idx is None:
        raise ValueError("Header not found for sourceplate1")
    header = raw.iloc[hdr_idx].tolist()
    df = pd.read_csv(path, skiprows=hdr_idx+1, header=None)
    df.columns = header
    df = df.dropna(how="all")
    seq_col = None
    for cand in ["Sequence with spaces", "Sequence (5' > 3')", "Sequence"]:
        if cand in df.columns:
            seq_col = cand; break
    if seq_col is None:
        seq_col = [c for c in df.columns if isinstance(c, str) and "sequence" in c.lower()][0]
    keep = ["Well Position", "Sequence Name", seq_col]
    df_keep = df[keep].copy()
    df_keep.rename(columns={"Well Position": "Well", seq_col: "Sequence"}, inplace=True)
    return df_keep, plate_name


def parse_source2(path: str) -> Tuple[pd.DataFrame, str]:
    """
    Robustly parse Source2-style replacement CSVs:
      - Preserve the source 'Well' column (replacement plate's well)
      - Normalize 'Replace Well' target column, preferring explicit name
      - Avoid misidentifying 'Well' as 'Replace Well'
    """
    raw = pd.read_csv(path, header=None)

    # --------- Plate name (optional preamble like "Plate SourcePlate2[1]") ----------
    plate_name = None
    for i in range(5):
        if i < len(raw):
            joined = " ".join(raw.iloc[i].dropna().astype(str))
            if joined.strip().startswith("Plate "):
                plate_name = joined.strip().replace("Plate ", "")
                break
    if not plate_name:
        plate_name = "SourcePlate2[1]"

    # ----------------------------- Find the header row -----------------------------
    hdr_idx = None
    for i in range(len(raw)):
        row = raw.iloc[i].astype(str).str.strip().str.upper()
        if "WELL" in row.values and "NAME" in row.values:
            hdr_idx = i; break
    if hdr_idx is None:
        raise ValueError("Header not found for sourceplate2")

    header = raw.iloc[hdr_idx].tolist()
    df = pd.read_csv(path, skiprows=hdr_idx+1, header=None)
    df.columns = header
    df = df.dropna(how="all")

    # ---------------------------- Helpers & normalizers ----------------------------
    def norm(s):
        return s.strip().lower() if isinstance(s, str) else ""

    colmap = {norm(c): c for c in df.columns if isinstance(c, str)}

    def get_col(*names, required=False):
        for n in names:
            if n in colmap:
                return colmap[n]
        if required:
            raise ValueError(f"Required column not found. Tried: {names}")
        return None

    # ---------------------------- Locate standard columns --------------------------
    name_col = get_col("name", "sequence name", required=True)

    seq_col = None
    for cand in ("sequence", "sequence with spaces", "sequence (5' > 3')"):
        if cand in colmap:
            seq_col = colmap[cand]; break
    if seq_col is None:
        seq_candidates = [c for c in df.columns if isinstance(c, str) and "sequence" in c.lower()]
        if not seq_candidates:
            raise ValueError("Could not find a sequence column in Source2 file.")
        seq_col = seq_candidates[0]

    # Source-plate 'Well' column (the well that holds the replacement sample)
    well_col = get_col("well", "well position", "source well", required=True)

    # --------------------- Locate the *Replace Well* target column -----------------
    replace_col = get_col("replace well", "replacement well", "replace_well", "replacewell")

    if replace_col is None:
        # heuristic
        well_pat = re.compile(r"^[A-Pa-p](0[1-9]|1[0-9]|2[0-4])$")  # A01–P24
        avoid = {norm(well_col), "well", "well position", "source well", "source well position"}
        candidates = []
        for c in df.columns:
            if not isinstance(c, str) or norm(c) in avoid:
                continue
            ser = df[c].dropna().astype(str).str.strip()
            if not ser.empty and ser.str.match(well_pat).mean() > 0.5:
                candidates.append((df.columns.get_loc(c), c))
        if candidates:
            candidates.sort(key=lambda t: t[0])
            replace_col = candidates[-1][1]

    if replace_col is None:
        raise ValueError("Could not locate a 'Replace Well' column in Source2.")

    # ----------------------------- Build normalized frame --------------------------
    keep = [well_col, name_col, seq_col, replace_col]
    df_keep = df[keep].copy()
    df_keep.rename(columns={
        well_col: "Well",          # preserve as source-plate well
        name_col: "Name",
        seq_col: "Sequence",
        replace_col: "Replace Well"
    }, inplace=True)

    # ------------------------------- Sanity checks --------------------------------
    expected = {"Well", "Name", "Sequence", "Replace Well"}
    missing = expected - set(df_keep.columns)
    if missing:
        raise ValueError(f"Source2 columns missing after normalization: {sorted(missing)}")

    if df_keep["Well"].equals(df_keep["Replace Well"]):
        raise ValueError(
            "Parsed 'Well' and 'Replace Well' columns are identical. "
            "Likely mis-detected 'Replace Well'."
        )

    return df_keep, plate_name

# ===================== Selection mapping helper (KEY FIX) ====================

def _rows_for_selected(s2_df: pd.DataFrame, selected_names: List[str]) -> pd.DataFrame:
    """
    Robustly map UI selections (e.g. 'U-PAINT_H01-01') onto rows in the Source2 CSV.

    Tries, in order:
      1) exact match against Name
      2) match against Replace Well using extracted targets like 'H01-01' / 'H01_01'
      3) match against Well (rare, but sometimes useful)
    """
    if not selected_names:
        return s2_df.iloc[0:0].copy()

    name_series = s2_df["Name"].astype(str).str.strip()
    rep_series  = s2_df["Replace Well"].astype(str).str.strip()
    well_series = s2_df["Well"].astype(str).str.strip()

    sel_set = set(str(x).strip() for x in selected_names)

    # 1) Direct match to Name (works if your button tooltip strings match CSV Name)
    m1 = s2_df[name_series.isin(sel_set)]
    if not m1.empty:
        return m1.copy()

    # 2) Interpret selection strings as targets: 'TAG_H01-01' -> 'H01-01'
    targets = set()
    for s in selected_names:
        s = str(s).strip()
        tail = s.split("_", 1)[1] if "_" in s else s
        tail = tail.strip()
        if not tail:
            continue
        targets.add(tail.upper())
        targets.add(tail.replace("-", "_").upper())
        targets.add(tail.replace("_", "-").upper())

    rep_up  = rep_series.str.upper()
    name_up = name_series.str.upper()
    well_up = well_series.str.upper()

    m2 = s2_df[rep_up.isin(targets)]
    if not m2.empty:
        return m2.copy()

    # 3) Last resort: match those targets against Name or Well too
    m3 = s2_df[name_up.isin(targets) | well_up.isin(targets)]
    return m3.copy()

# ======================== Picklist builder (combined) ========================

def generate_picklist_combined(
    base_source1_path: str,
    replacements: List[Tuple[str, List[str], str]],  # [(source2_path, selected_names, plate_name_override), ...]
    dest_wells: List[str],
    source_plate_type: str = "384PP_AQ_BP",
    destination_plate_name: str = "Destination[1]",
    transfer_volume_nL: int = 50,
    max_dest_well_volume_uL: float = 12.5,
    transfers_per_source: int = 1,
) -> pd.DataFrame:
    if not dest_wells:
        raise ValueError("dest_wells must be non-empty.")

    s1_df, s1_name = parse_source1(base_source1_path)

    chosen_rows: List[pd.DataFrame] = []
    replace_targets: Dict[str, Tuple[str, str]] = {}  # Replace Well -> (source2_path, name)

    for s2_path, selected_names, s2_override_name in replacements:
        if not selected_names:
            continue
        s2_df, s2_name_from_file = parse_source2(s2_path)
        s2_name = s2_override_name or s2_name_from_file

        # >>>>>>> KEY CHANGE: robust selection mapping (fixes Set D/E) <<<<<<<
        matched = _rows_for_selected(s2_df, selected_names)
        if matched.empty:
            # You can comment this out once you're confident it's working
            # print("WARNING: nothing matched in", s2_path, "example selections:", selected_names[:5])
            continue

        # Prevent conflicting replacements for same target well (across all sets)
        for _, r in matched.iterrows():
            tgt = str(r["Replace Well"]).upper().strip()
            nm  = str(r["Name"])
            if tgt in replace_targets:
                prev_path, prev_nm = replace_targets[tgt]
                raise ValueError(
                    f"Duplicate replacement for Replace Well '{tgt}': "
                    f"'{nm}' in {s2_path} conflicts with '{prev_nm}' in {prev_path}"
                )
            replace_targets[tgt] = (s2_path, nm)

        matched["Source Plate Name"] = s2_name
        matched["Source Plate Type"] = source_plate_type
        chosen_rows.append(matched[["Well", "Sequence", "Source Plate Name", "Source Plate Type", "Replace Well"]])

    # Start with Source1 (base) actives
    active = s1_df.copy()
    active["Source Plate Name"] = s1_name
    active["Source Plate Type"] = source_plate_type
    active = active[["Well", "Sequence", "Source Plate Name", "Source Plate Type"]]

    # Apply replacements (drop replaced base wells; add chosen replacement wells)
    if chosen_rows:
        s2_all = pd.concat(chosen_rows, ignore_index=True)
        to_drop = set(s2_all["Replace Well"].astype(str).str.upper())
        active = active[~active["Well"].astype(str).str.upper().isin(to_drop)]
        s2_active = s2_all[["Well", "Sequence", "Source Plate Name", "Source Plate Type"]].copy()
        active_sources = pd.concat([active, s2_active], ignore_index=True)
    else:
        active_sources = active

    # Build transfers (one or more per source)
    transfers = []
    for _, r in active_sources.iterrows():
        for _ in range(transfers_per_source):
            transfers.append({
                "Source Plate Name": r["Source Plate Name"],
                "Source Plate Type": r["Source Plate Type"],
                "Source Well": str(r["Well"]).upper(),
                "Sample Comments": r.get("Sequence", ""),
            })
    transfers_df = pd.DataFrame(transfers)

    # Capacity & destination assignment
    max_nL = int(max_dest_well_volume_uL * 1000)
    per_well_capacity = max_nL // transfer_volume_nL
    total_transfers = len(transfers_df)
    max_capacity = per_well_capacity * len(dest_wells)
    if total_transfers > max_capacity:
        raise ValueError(
            f"Too many transfers ({total_transfers}) for provided destination wells "
            f"(max {max_capacity} at {transfer_volume_nL} nL each)."
        )
    dest_indices = [i // per_well_capacity for i in range(total_transfers)]
    dest_wells_assigned = [dest_wells[i] for i in dest_indices]

    transfers_df["Destination Plate Name"] = destination_plate_name
    transfers_df["Destination Well"] = dest_wells_assigned
    transfers_df["Transfer Volume"] = transfer_volume_nL

    pick = transfers_df[[
        "Source Plate Name", "Source Plate Type", "Source Well",
        "Sample Comments", "Destination Plate Name", "Destination Well",
        "Transfer Volume"
    ]].copy()
    return pick

# ================= Reagent mixing (your function, unchanged) =================

def give_volumes(staple, num_staple, scaffold, mg, na, xte_10, des):
    initial_vol = des[0]
    conc_staple = staple[0]/des[1]
    vol_staple = (initial_vol/conc_staple)*num_staple
    carry_xte_staple = vol_staple*staple[1]

    conc_scaffold = scaffold[0]/des[2]
    vol_scaffold = initial_vol/conc_scaffold
    carry_xte_scaffold = vol_scaffold*scaffold[1]

    conc_xte = xte_10[1]/des[3]
    vol_xte = (initial_vol - carry_xte_scaffold - carry_xte_staple)/conc_xte

    conc_mg = mg[2]/des[4]
    vol_mg = initial_vol/conc_mg

    conc_na = na[3]/des[5]
    vol_na = initial_vol/conc_na

    vol_water = initial_vol - vol_staple - vol_scaffold - vol_xte - vol_mg
    return (vol_staple, vol_scaffold, vol_xte, vol_mg, vol_na, vol_water)

# ===================== Grid builders (dense / sparse) ========================

def build_dense_grid(tag: str,
                     row_labels: List[str],
                     col_labels: List[str],
                     divider_after: Optional[str],
                     show_headers: bool,
                     disabled_positions: Optional[Set[Tuple[str, str]]] = None,
                     enabled_positions: Optional[Set[Tuple[str, str]]] = None,
                     color_map: Optional[Dict[Tuple[str, str], str]] = None,
                     cell_size_px: int = 32,
                     row_spacing_px: int = 2,
                     col_spacing_px: int = 2,
                     divider_spacing_px: int = 8):

    disabled_positions = disabled_positions or set()
    color_map = color_map or {}

    buttons_by_name = {}
    col_boxes = []
    css_rules = []

    def _safe_class(s: str) -> str:
        return re.sub(r"[^a-zA-Z0-9_-]", "_", s)

    # ---- Row labels ----
    row_label_widgets = [W.Label("") if show_headers else W.HTML("")]
    for rlab in row_labels:
        row_label_widgets.append(
            W.HTML(
                f"<b>{rlab}</b>",
                layout=W.Layout(
                    width="60px",
                    height=f"{cell_size_px}px",
                    display="flex",
                    align_items="center",
                    justify_content="flex-end",
                    margin=f"{row_spacing_px}px 0px"
                )
            )
        )
    left_labels = W.VBox(row_label_widgets, layout=W.Layout(margin="0 6px 0 0"))

    # ---- Build columns ----
    for c in col_labels:

        header = W.HTML(
            f"<b>{c}</b>" if show_headers else "",
            layout=W.Layout(
                width=f"{cell_size_px}px",
                height="24px" if show_headers else "0px",
                display="flex",
                align_items="center",
                justify_content="center"
            )
        )

        cells = [header]

        for rlab in row_labels:
            name = f"{tag}_{rlab}-{c}"
            pos = (rlab, c)

            if enabled_positions is not None:
                is_disabled = pos not in enabled_positions
            else:
                is_disabled = pos in disabled_positions

            b = W.ToggleButton(
                description="",
                tooltip=name,
                disabled=is_disabled,
                layout=W.Layout(
                    width=f"{cell_size_px}px",
                    height=f"{cell_size_px}px",
                    padding="0px",
                    margin=f"{row_spacing_px}px 0px"
                )
            )

            # ------------------------------
            # Colored wells = border only
            # ------------------------------
            if (not is_disabled) and (pos in color_map):
                cls = _safe_class(f"well_{tag}_{rlab}_{c}")
                b.add_class(cls)
                col = str(color_map[pos]).strip()

                css_rules.append(
                    f".{cls}, button.{cls}, .{cls} button, .{cls} > button "
                    f"{{ border: 2px solid {col} !important; }}"
                )
                css_rules.append(
                    f".{cls}.mod-active, button.{cls}.mod-active, "
                    f".{cls}[aria-pressed='true'], button.{cls}[aria-pressed='true'] "
                    f"{{ border: 4px solid {col} !important; }}"
                )

            # ------------------------------
            # Disabled wells = greyed out
            # ------------------------------
            if is_disabled:
                dis_cls = _safe_class(f"disabled_{tag}_{rlab}_{c}")
                b.add_class(dis_cls)
                css_rules.append(
                    f".{dis_cls}, button.{dis_cls}, .{dis_cls} button, .{dis_cls} > button "
                    f"{{ background-color: #e6e6e6 !important; border: 1px solid #bdbdbd !important; }}"
                )

            buttons_by_name[name] = b
            cells.append(b)

        col_boxes.append(
            W.VBox(
                cells,
                layout=W.Layout(
                    align_items="center",
                    margin=f"0px {col_spacing_px}px"
                )
            )
        )

        if divider_after and c == divider_after:
            col_boxes.append(
                W.HTML(
                    "",
                    layout=W.Layout(
                        width="2px",
                        background_color="black",
                        margin=f"0px {divider_spacing_px}px"
                    )
                )
            )

    css_widget = W.HTML("<style>\n" + "\n".join(css_rules) + "\n</style>") if css_rules else W.HTML("")

    grid = W.VBox([
        css_widget,
        W.HBox([left_labels] + col_boxes, layout=W.Layout(margin="6px 0"))
    ])

    return buttons_by_name, grid


def build_sparse_grid(tag: str,
                      row_labels: List[str],
                      col_labels: List[str],
                      active_positions: Set[Tuple[str, str]],
                      divider_after: Optional[str],
                      show_headers: bool):
    buttons_by_name = {}
    col_boxes = []

    row_label_widgets = [W.Label("") if show_headers else W.HTML("")]
    for rlab in row_labels:
        row_label_widgets.append(W.HTML(
            f"<b>{rlab}</b>",
            layout=W.Layout(width="60px", height="32px",
                            display="flex", align_items="center", justify_content="flex-end")
        ))
    left_labels = W.VBox(row_label_widgets, layout=W.Layout(margin="0 6px 0 0"))

    for c in col_labels:
        header = W.HTML(f"<b>{c}</b>" if show_headers else "",
                        layout=W.Layout(width="40px", height=("24px" if show_headers else "0px"),
                                        display="flex", align_items="center", justify_content="center"))
        cells = [header]
        for rlab in row_labels:
            if (rlab, c) in active_positions:
                name = f"{tag}_{rlab}-{c}"
                b = W.ToggleButton(description="", tooltip=name,
                                   layout=W.Layout(width="32px", height="32px", padding="0px", margin="1px"))
                buttons_by_name[name] = b
                cells.append(b)
            else:
                cells.append(W.HTML("", layout=W.Layout(width="32px", height="32px", margin="1px")))
        col_boxes.append(W.VBox(cells, layout=W.Layout(align_items="center")))

        if divider_after and c == divider_after:
            height = (24 if show_headers else 0) + len(row_labels) * 34
            col_boxes.append(W.HTML("", layout=W.Layout(width="2px", height=f"{height}px",
                                                        background_color="black", margin="0 8px")))

    grid = W.HBox([left_labels] + col_boxes, layout=W.Layout(margin="6px 0"))
    return buttons_by_name, grid

# ================= Reusable "Set" builder with many panels ===================

DESC_W    = "220px"
INPUT_W   = "700px"
INPUT_W_S = "260px"

def mk_text(value, description, width=INPUT_W, desc_w=DESC_W):
    return W.Text(value=value, description=description,
                  layout=W.Layout(width=width),
                  style={'description_width': desc_w})

def mk_int(value, description, width=INPUT_W_S, desc_w="120px"):
    return W.IntText(value=value, description=description,
                     layout=W.Layout(width=width),
                     style={'description_width': desc_w})

def mk_float(value, description, width=INPUT_W_S, desc_w="120px"):
    return W.FloatText(value=value, description=description,
                       layout=W.Layout(width=width),
                       style={'description_width': desc_w})

def build_replacement_set(
    set_title: str,
    source2_default: str,
    panels: List[Dict],
    plate_name_override: str,
):
    source2_path = W.Text(
        value=source2_default,
        description=f"{set_title} Source2:",
        layout=W.Layout(width=INPUT_W),
        style={'description_width': DESC_W}
    )
    status = W.HTML(value="")

    children = []
    button_dicts = []

    for p in panels:
        label = p["label"]
        layout_kind = p.get("layout", "dense")
        rows = p["row_labels"]
        cols = p["col_labels"]
        divider_after = p.get("divider_after", None)
        show_headers = p.get("show_headers", True)

        disabled_positions = p.get("disabled_positions", set())
        enabled_positions  = p.get("enabled_positions", None)
        color_map          = p.get("color_map", {})

        cell_size_px       = int(p.get("cell_size_px", 32))
        row_spacing_px     = int(p.get("row_spacing_px", 2))
        col_spacing_px     = int(p.get("col_spacing_px", 2))
        divider_spacing_px = int(p.get("divider_spacing_px", 8))

        if layout_kind == "dense":
            btns, grid = build_dense_grid(
                label, rows, cols, divider_after, show_headers,
                disabled_positions=disabled_positions,
                enabled_positions=enabled_positions,
                color_map=color_map,
                cell_size_px=cell_size_px,
                row_spacing_px=row_spacing_px,
                col_spacing_px=col_spacing_px,
                divider_spacing_px=divider_spacing_px
            )
        else:
            btns, grid = build_sparse_grid(label, rows, cols, p["active_positions"], divider_after, show_headers)

        button_dicts.append(btns)
        box = W.VBox([W.HTML(f"<b>{label}</b>"), grid], layout=W.Layout(margin="4px 0 10px 0"))
        children.append(box)

    acc = W.Accordion(children=children)
    for i, p in enumerate(panels):
        acc.set_title(i, p["label"])

    container = W.VBox([
        W.HTML(f"<h3 style='margin:4px 0'>{set_title}: choose replacements in any panel(s)</h3>"),
        acc,
        W.HBox([source2_path]),
        status
    ])

    def get_selected_names() -> List[str]:
        selected = []
        for d in button_dicts:
            selected += [nm for nm, b in d.items() if b.value]
        return selected

    return {
        "widget": container,
        "get_source2_path": lambda: source2_path.value,
        "get_selected_names": get_selected_names,
        "set_status": lambda html: setattr(status, "value", html),
        "get_plate_name_override": lambda: plate_name_override,
    }

# ====================== Define panels for A / B / C / D / E ==========================

rows_A = ["H01","H03","H05","H07","H09","H11","H13","H15"]
cols_A = [f"{i:02d}" for i in range(1,13)]
panels_A = [
    {"label": "D-Apt", "layout": "dense", "row_labels": rows_A, "col_labels": cols_A, "divider_after": "06", "show_headers": True, "row_spacing_px": 1, "col_spacing_px": 12, "divider_spacing_px": 20, "cell_size_px": 20},
    {"label": "U-Apt", "layout": "dense", "row_labels": rows_A, "col_labels": cols_A, "divider_after": "06", "show_headers": True, "row_spacing_px": 1, "col_spacing_px": 12, "divider_spacing_px": 20, "cell_size_px": 20},
]
setA = build_replacement_set("Set A", SOURCE2_A_DEFAULT, panels_A, plate_name_override="SourcePlate3[3]")

rows_B = ["R1","R2"]
cols_B = [f"{i:02d}" for i in range(1,7)]
panels_B = [
    {"label": "D-Orthogonal", "layout": "dense", "row_labels": rows_B, "col_labels": cols_B, "divider_after": "03", "show_headers": True},
    {"label": "U-Orthogonal", "layout": "dense", "row_labels": rows_B, "col_labels": cols_B, "divider_after": "03", "show_headers": True},
    {"label": "D-Identical",  "layout": "dense", "row_labels": rows_B, "col_labels": cols_B, "divider_after": "03", "show_headers": True},
    {"label": "U-Identical",  "layout": "dense", "row_labels": rows_B, "col_labels": cols_B, "divider_after": "03", "show_headers": True},
]
setB = build_replacement_set("Set B", SOURCE2_B_DEFAULT, panels_B, plate_name_override="SourcePlate2[2]")

rows_C = ["Top", "Mid", "Bot"]
cols_C = ["L3", "L2", "L1", "R1", "R2", "R3"]
active_C_left = {("Top","L3"), ("Mid","L3"), ("Bot","L3"), ("Mid","L2"), ("Mid","L1")}
active_C_right = {("Top","R3"), ("Mid","R3"), ("Bot","R3"), ("Mid","R2"), ("Mid","R1")}
left_panel_labels  = ["PDGF-Apt", "Kana-Apt"]
right_panel_labels = ["PDGF-14","PDGF-18","PDGF-22","PDGF-26","PDGF-30","PDGF-34","PDGF-38",
                      "Kana-14","Kana-18","Kana-22"]

panels_C: List[Dict] = []
for lbl in left_panel_labels:
    panels_C.append({"label": lbl, "layout": "sparse", "row_labels": rows_C, "col_labels": cols_C,
                     "active_positions": active_C_left, "divider_after": "L1", "show_headers": False})
for lbl in right_panel_labels:
    panels_C.append({"label": lbl, "layout": "sparse", "row_labels": rows_C, "col_labels": cols_C,
                     "active_positions": active_C_right, "divider_after": "L1", "show_headers": False})
setC = build_replacement_set("Set C", SOURCE2_C_DEFAULT, panels_C, plate_name_override="SourcePlate4[4]")

rows_D = ["H01","H03","H05","H07","H09","H11","H13","H15"]
cols_D = [f"{i:02d}" for i in range(1,13)]
enabled_D = {
    ("H01", "01"), ("H01", "12"),
    ("H05", "04"), ("H05", "09"),
    ("H11", "04"), ("H11", "09"),
    ("H15", "02"), ("H15", "12"),
    ("H07", "04"), ("H07", "09"),
    ("H13", "04"), ("H13", "09"),
}
enabled_U = {("H05", "04"), ("H05", "09"), ("H11", "04"), ("H11", "09")}

panels_D = [
    {"label": "D-MB", "layout": "dense", "row_labels": rows_D, "col_labels": cols_D,
     "divider_after": "06", "show_headers": True, "enabled_positions": enabled_D,
     "row_spacing_px": 1, "col_spacing_px": 12, "divider_spacing_px": 20, "cell_size_px": 20},
    {"label": "U-MB", "layout": "dense", "row_labels": rows_D, "col_labels": cols_D,
     "divider_after": "06", "show_headers": True, "enabled_positions": enabled_U,
     "row_spacing_px": 1, "col_spacing_px": 12, "divider_spacing_px": 20, "cell_size_px": 20},
]
setD = build_replacement_set("Set D", SOURCE2_D_DEFAULT, panels_D, plate_name_override="SourcePlate4[4]")

rows_E = ["H01","H03","H05","H07","H09","H11","H13","H15"]
cols_E = [f"{i:02d}" for i in range(1,13)]

enabled_E = {
    ("H01", "01"), ("H01", "05"), ("H01", "08"), ("H01", "12"),
    ("H15", "01"), ("H15", "05"), ("H15", "08"), ("H15", "12"),
}

color_E = {
    ("H01","01"): "red",   ("H01","02"): "blue",  ("H01","03"): "green",
    ("H01","04"): "red",   ("H01","05"): "blue",  ("H01","06"): "green",
    ("H01","07"): "red",   ("H01","08"): "blue",  ("H01","09"): "green",
    ("H01","10"): "red",   ("H01","11"): "blue",  ("H01","12"): "green",

    ("H03","01"): "green", ("H03","02"): "red",   ("H03","03"): "blue",
    ("H03","04"): "green", ("H03","05"): "red",   ("H03","06"): "blue",
    ("H03","07"): "green", ("H03","08"): "red",   ("H03","09"): "blue",
    ("H03","10"): "green", ("H03","11"): "red",   ("H03","12"): "blue",

    ("H05","01"): "blue",  ("H05","02"): "green", ("H05","03"): "red",
    ("H05","04"): "blue",  ("H05","05"): "green", ("H05","06"): "red",
    ("H05","07"): "blue",  ("H05","08"): "green", ("H05","09"): "red",
    ("H05","10"): "blue",  ("H05","11"): "green", ("H05","12"): "red",

    ("H07","01"): "red",   ("H07","02"): "blue",  ("H07","03"): "green",
    ("H07","04"): "red",   ("H07","05"): "blue",  ("H07","06"): "green",
    ("H07","07"): "red",   ("H07","08"): "blue",  ("H07","09"): "green",
    ("H07","10"): "red",   ("H07","11"): "blue",  ("H07","12"): "green",

    ("H09","01"): "green", ("H09","02"): "red",   ("H09","03"): "blue",
    ("H09","04"): "green", ("H09","05"): "red",   ("H09","06"): "blue",
    ("H09","07"): "green", ("H09","08"): "red",   ("H09","09"): "blue",
    ("H09","10"): "green", ("H09","11"): "red",   ("H09","12"): "blue",

    ("H11","01"): "blue",  ("H11","02"): "green", ("H11","03"): "red",
    ("H11","04"): "blue",  ("H11","05"): "green", ("H11","06"): "red",
    ("H11","07"): "blue",  ("H11","08"): "green", ("H11","09"): "red",
    ("H11","10"): "blue",  ("H11","11"): "green", ("H11","12"): "red",

    ("H13","01"): "red",   ("H13","02"): "blue",  ("H13","03"): "green",
    ("H13","04"): "red",   ("H13","05"): "blue",  ("H13","06"): "green",
    ("H13","07"): "red",   ("H13","08"): "blue",  ("H13","09"): "green",
    ("H13","10"): "red",   ("H13","11"): "blue",  ("H13","12"): "green",

    ("H15","01"): "green", ("H15","02"): "red",   ("H15","03"): "blue",
    ("H15","04"): "green", ("H15","05"): "red",   ("H15","06"): "blue",
    ("H15","07"): "green", ("H15","08"): "red",   ("H15","09"): "blue",
    ("H15","10"): "green", ("H15","11"): "red",   ("H15","12"): "blue",
}

panels_E = [
    {"label": "D-Biotin", "layout": "dense", "row_labels": rows_E, "col_labels": cols_E, "divider_after": "06", "show_headers": True,
     "enabled_positions": enabled_E, "row_spacing_px": 1, "col_spacing_px": 12, "divider_spacing_px": 20, "cell_size_px": 20},
    {"label": "U-PAINT", "layout": "dense", "row_labels": rows_E, "col_labels": cols_E, "divider_after": "06", "show_headers": True,
     "row_spacing_px": 1, "col_spacing_px": 12, "divider_spacing_px": 20, "cell_size_px": 20, "color_map": color_E},
]
setE = build_replacement_set("Set E", SOURCE2_E_DEFAULT, panels_E, plate_name_override="SourcePlate5[5]")

# ================= Tabs (A / B / C / D / E) =========================================

tabs = W.Tab(children=[setA["widget"], setB["widget"], setC["widget"], setD["widget"], setE["widget"]])
tabs.set_title(0, "Set A")
tabs.set_title(1, "Set B")
tabs.set_title(2, "Set C")
tabs.set_title(3, "Set D")
tabs.set_title(4, "Set E")

# ================= Global controls + combined runner + MIXING UI =============

base_source1_path = mk_text(BASE_SOURCE1_DEFAULT, "Base Source1:")
dest_wells_text   = mk_text("A01,A02,A03", "Dest wells:")
out_path          = mk_text(OUT_COMBINED_DEFAULT, "Save picklist:")

transfers_per_source = mk_int(1, "Transfers/src:", width=INPUT_W_S, desc_w="120px")
transfer_vol         = mk_int(50, "Vol (nL):",     width=INPUT_W_S, desc_w="120px")
cap_ul               = mk_float(12.5, "Max µL/well:", width=INPUT_W_S, desc_w="120px")

dest_plate_name = mk_text("Destination[1]", "Dest plate:", width=INPUT_W_S, desc_w="120px")
src_plate_type  = mk_text("384PP_AQ_BP",   "Src type:",   width=INPUT_W_S, desc_w="120px")

mix_out_path = mk_text(OUT_MIXING_DEFAULT, "Save recipe:")

staple_input   = mk_text("200000,1,0,0",   "staple [nM,xTE,mg,na]:")
scaffold_input = mk_text("400,0.1,0,0",    "scaffold [...]:")
xte10_input    = mk_text("0,10,0,0",       "xte_10 [...]:")
mg_input       = mk_text("0,0,100,0",      "mg [...]:")
na_input       = mk_text("0,0,0,100",      "na [...]:")
des_input      = mk_text("500,10,1,1,12,5","des [µL,nM,nM,x,mM,mM]:")

run_all_btn = W.Button(description="Generate Picklist + Mixing Recipe",
                       button_style="success",
                       layout=W.Layout(width="360px"))
overall_status = W.HTML(value="")

def on_run_all_clicked(_):
    try:
        reps = []
        for s in (setA, setB, setC, setD, setE):
            names = s["get_selected_names"]()
            s2p = s["get_source2_path"]()
            override_name = s["get_plate_name_override"]()
            if names:
                reps.append((s2p, names, override_name))

        if not reps:
            overall_status.value = "<b style='color:#d9534f'>No selections made in any panel.</b>"
            return

        dest_list = [w.strip().upper() for w in dest_wells_text.value.split(",") if w.strip()]
        if not dest_list:
            overall_status.value = "<b style='color:#d9534f'>Please enter at least one destination well.</b>"
            return

        pick = generate_picklist_combined(
            base_source1_path=base_source1_path.value,
            replacements=reps,
            dest_wells=dest_list,
            source_plate_type=src_plate_type.value,
            destination_plate_name=dest_plate_name.value,
            transfer_volume_nL=int(transfer_vol.value),
            max_dest_well_volume_uL=float(cap_ul.value),
            transfers_per_source=int(transfers_per_source.value),
        )
        pick.to_csv(out_path.value, index=False)

        unique_sources = pick[["Source Plate Name", "Source Well"]].drop_duplicates()
        num_staple = int(unique_sources.shape[0])

        per_source_nL = int(transfer_vol.value) * int(transfers_per_source.value)
        available_staple_uL = (num_staple * per_source_nL) / 1000.0

        staple   = _parse_vec(staple_input.value,   4, "staple")
        scaffold = _parse_vec(scaffold_input.value, 4, "scaffold")
        xte_10   = _parse_vec(xte10_input.value,    4, "xte_10")
        mg       = _parse_vec(mg_input.value,       4, "mg")
        na       = _parse_vec(na_input.value,       4, "na")
        des      = _parse_vec(des_input.value,      6, "des")

        vol_staple, vol_scaffold, vol_xte, vol_mg, vol_na, vol_water = give_volumes(
            staple, num_staple, scaffold, mg, na, xte_10, des
        )

        if vol_staple > available_staple_uL + 1e-9:
            overall_status.value = (
                f"<b style='color:#d9534f'>Error:</b> Requested staple volume ({vol_staple:.3f} µL) "
                f"exceeds available from picklist ({available_staple_uL:.3f} µL). "
                f"Decrease 'des[0]' (final µL) or increase transfer volume / transfers per source."
            )
            display(pick.head(20))
            return

        mix_rows = [
            {"Reagent": "Staple mix (from picklist)", "Volume_uL": vol_staple},
            {"Reagent": "Scaffold",                    "Volume_uL": vol_scaffold},
            {"Reagent": "10X TE",                      "Volume_uL": vol_xte},
            {"Reagent": "Mg (100 mM)",                 "Volume_uL": vol_mg},
            {"Reagent": "Na (100 mM)",                 "Volume_uL": vol_na},
            {"Reagent": "DI Water",                    "Volume_uL": vol_water},
        ]
        mix_df = pd.DataFrame(mix_rows)
        mix_df.to_csv(mix_out_path.value, index=False)

        overall_status.value = (
            f"<b style='color:#5cb85c'>Success.</b> "
            f"Picklist: {_link(out_path.value)} (transfers: {len(pick)})<br>"
            f"Mixing recipe: {_link(mix_out_path.value)}<br>"
            f"Unique staples: <b>{num_staple}</b> | "
            f"Available staple volume: <b>{available_staple_uL:.3f} µL</b> | "
            f"Requested staple volume: <b>{vol_staple:.3f} µL</b>"
        )

        for s in (setA, setB, setC, setD, setE):
            s["set_status"]("")

        display(Markdown("**Picklist (first 20 rows):**"))
        display(pick.head(20))
        display(Markdown("**Mixing recipe:**"))
        display(mix_df)

    except Exception as e:
        overall_status.value = f"<b style='color:#d9534f'>Error:</b> {e!s}"

run_all_btn.on_click(on_run_all_clicked)

# ============================= Render UI =====================================

display(Markdown("### Echo picklist GUI + Mixing Recipe\n"
                 "- Set A: D-Apt, U-Apt\n"
                 "- Set B: D-Orthogonal, U-Orthogonal, D-Identical, U-Identical\n"
                 "- Set C: Left = PDGF-Apt, Kana-Apt; Right = PDGF-14, -18, -22, -26, -30, -34, -38; Kana-14, -18, -22\n"
                 "\nSelect any replacements, then click **Generate Picklist + Mixing Recipe**."))

display(tabs)

display(W.VBox([
    W.HTML("<hr>"),
    W.HBox([base_source1_path]),
    W.HBox([dest_wells_text]),
    W.HBox([transfers_per_source, transfer_vol, cap_ul]),
    W.HBox([dest_plate_name, src_plate_type]),
    W.HBox([out_path]),
    W.HTML("<b>Mixing parameters</b>"),
    W.HBox([mix_out_path]),
    W.HBox([staple_input]),
    W.HBox([scaffold_input]),
    W.HBox([xte10_input]),
    W.HBox([mg_input]),
    W.HBox([na_input]),
    W.HBox([des_input]),
    run_all_btn,
    overall_status
]))


### Echo picklist GUI + Mixing Recipe
- Set A: D-Apt, U-Apt
- Set B: D-Orthogonal, U-Orthogonal, D-Identical, U-Identical
- Set C: Left = PDGF-Apt, Kana-Apt; Right = PDGF-14, -18, -22, -26, -30, -34, -38; Kana-14, -18, -22

Select any replacements, then click **Generate Picklist + Mixing Recipe**.

In [15]:
df_concat = concat_picklists(
    input_paths=[
        r"/Users/maxladabaum/Desktop/origami_book/picklists/picklist_06_03_26__1.csv",
        r"/Users/maxladabaum/Desktop/origami_book/picklists/picklist_06_03_26__2.csv",
        r"/Users/maxladabaum/Desktop/origami_book/picklists/picklist_06_03_26__3.csv",
        r"/Users/maxladabaum/Desktop/origami_book/picklists/picklist_06_03_26__4.csv",
        r"/Users/maxladabaum/Desktop/origami_book/picklists/picklist_06_03_26__5.csv",
        r"/Users/maxladabaum/Desktop/origami_book/picklists/picklist_06_03_26__6.csv",
        r"/Users/maxladabaum/Desktop/origami_book/picklists/picklist_06_03_26__7.csv",
        r"/Users/maxladabaum/Desktop/origami_book/picklists/picklist_06_03_26__8.csv",
        r"/Users/maxladabaum/Desktop/origami_book/picklists/picklist_06_03_26__9.csv",
        r"/Users/maxladabaum/Desktop/origami_book/picklists/picklist_06_03_26__10.csv",
        r"/Users/maxladabaum/Desktop/origami_book/picklists/picklist_06_03_26__11.csv",
        r"/Users/maxladabaum/Desktop/origami_book/picklists/picklist_06_03_26__12.csv",
    ],
    save_path=r"/Users/maxladabaum/Desktop/origami_book/picklists/06_03_26_paint.csv",
    dedupe=True,
    sort=True,
    keep_provenance=True
)
display(df_concat.head())

=== Picklist Concatenation Summary ===
        picklist_06_03_26__1.csv : 282 rows
        picklist_06_03_26__2.csv : 282 rows
        picklist_06_03_26__3.csv : 282 rows
        picklist_06_03_26__4.csv : 282 rows
        picklist_06_03_26__5.csv : 282 rows
        picklist_06_03_26__6.csv : 282 rows
        picklist_06_03_26__7.csv : 282 rows
        picklist_06_03_26__8.csv : 282 rows
        picklist_06_03_26__9.csv : 282 rows
       picklist_06_03_26__10.csv : 282 rows
       picklist_06_03_26__11.csv : 282 rows
       picklist_06_03_26__12.csv : 282 rows
--------------------------------------
   Total input rows          : 3,384
   Removed duplicate rows    : 0
   Final concatenated rows   : 3,384
   Saved to                  : /Users/maxladabaum/Desktop/origami_book/picklists/06_03_26_paint.csv


/var/folders/9k/zcxy0pgj4dbgsxn2hc8q99j40000gn/T/ipykernel_81563/974835074.py:73: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/var/folders/9k/zcxy0pgj4dbgsxn2hc8q99j40000gn/T/ipykernel_81563/974835074.py:73: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/var/folders/9k/zcxy0pgj4dbgsxn2hc8q99j40000gn/T/ipykernel_81563/974835074.py:73: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/var/folders/9k/zcxy0pgj4dbgsxn2hc8q99j40000gn/T/ipykernel_81563/974835074.py:73: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/var/folders/9k/zcxy0pgj4dbgsxn2hc8q99j40000gn/T/ipykernel_81563/974

,Source Plate Name,Source Plate Type,Source Well,Destination Plate Name,Destination Well,Transfer Volume,Sample Comments,InputFile
0,SourcePlate1[1],384PP_AQ_BP,A01,Destination[1],B01,25,TTT TTA TAA CCG ATA TAT TCA ACC ATC GCC CAC GC...,picklist_06_03_26__1.csv
1,SourcePlate1[1],384PP_AQ_BP,A02,Destination[1],B01,25,TTT TTC GGG CGC GGT TGC GGT GTA AAG CCT GGG GT...,picklist_06_03_26__1.csv
2,SourcePlate1[1],384PP_AQ_BP,A03,Destination[1],B01,25,TCC GGT ATC CCA CAA GAA TTG AGT TTA AGA AA,picklist_06_03_26__1.csv
3,SourcePlate1[1],384PP_AQ_BP,A04,Destination[1],B01,25,CGA ACC TCG GGT AAT TGA GCG CTT ACC AGA,picklist_06_03_26__1.csv
4,SourcePlate1[1],384PP_AQ_BP,A05,Destination[1],B01,25,GGT GTG TGT GTT TCT TAA ATC AGA ATT AAC TGA AC...,picklist_06_03_26__1.csv
